# 05 — Null Model Calibration & Power Analysis

**Goal**: Rigorously answer the question: *Can persistent homology detect
sub-threshold astrophysical populations in forced photometry data?*

We perform:
1. **Null distribution construction** — build the empirical distribution of
   topological summary statistics from pure-noise ensembles
2. **Power analysis** — inject known populations at varying (SNR, population size)
   and measure detection rates
3. **Sensitivity curves** — minimum population size for reliable detection at each SNR

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from void.embedding.takens import TakensEmbedder
from void.topology.null_model import build_null_distribution, compare_ensemble_to_null
from void.analysis.anomaly import power_analysis
from void.data.synthetic import generate_ensemble, generate_periodic, generate_transient

plt.rcParams.update({"figure.dpi": 120, "font.size": 10})
rng = np.random.default_rng(42)

## 1. Build the null distribution

Generate 100 noise-only ensembles of 250 sources each, compute persistence
on each, and build the empirical null. This is the reference against which
all observed ensembles will be compared.

**Note**: This cell takes a few minutes to run. Reduce `n_realizations` for a quick test.

In [ ]:
embedder = TakensEmbedder(dimension=3, delay=2)

null_dist = build_null_distribution(
    n_realizations=100,
    n_sources_per=250,
    n_epochs=200,
    embedder=embedder,
    maxdim=1,
    rng=np.random.default_rng(rng.integers(0, 2**63)),
    verbose=True,
)

print(f"\nNull distribution: {null_dist.n_realizations} realizations")
print(f"Statistics tracked: {null_dist.stat_names}")
print(f"Means: {null_dist.mean()}")
print(f"Stds:  {null_dist.std()}")

In [ ]:
# Visualize the null distributions
fig, axes = plt.subplots(2, 4, figsize=(16, 6))
for i, (name, ax) in enumerate(zip(null_dist.stat_names, axes.ravel())):
    vals = null_dist.summary_stats[:, i]
    ax.hist(vals, bins=20, edgecolor="black", alpha=0.7)
    ax.axvline(np.mean(vals), color="red", linestyle="--", label="mean")
    ax.set_title(name, fontsize=9)
    ax.legend(fontsize=7)

fig.suptitle("Null Distribution of Topological Summary Statistics", y=1.02)
fig.tight_layout()
plt.show()

## 2. Sanity check: test a known-signal ensemble

Before the full power analysis, verify that a strong signal is detected.

In [ ]:
# Strong signal: 50 periodic sources at SNR=5
strong_ensemble = generate_ensemble(
    n_signal=50, n_noise=200,
    signal_generator=generate_periodic,
    signal_kwargs={"snr": 5.0, "period": 30.0, "n_epochs": 200},
    noise_kwargs={"n_epochs": 200},
    rng=np.random.default_rng(rng.integers(0, 2**63)),
)

result = compare_ensemble_to_null(strong_ensemble, null_dist, embedder=embedder)

print("Test result (strong signal):")
print(f"  Significant: {result['significant']}")
print(f"  Most significant: {result['most_significant_stat']} (p={result['min_p_value']:.4f})")
for name, p, z in zip(result['stat_names'], result['p_values'], result['z_scores']):
    flag = " ***" if p < 0.05 else ""
    print(f"  {name:30s}: p={p:.4f}, z={z:+.2f}{flag}")

## 3. Power analysis: periodic signals

Sweep over SNR (1-5σ) and population sizes (10-200 sources) to measure
detection rates. This produces the key sensitivity curves.

**Note**: This is the most computationally expensive cell. Reduce ranges for quick iteration.

In [ ]:
pa_results = power_analysis(
    snr_levels=[1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 5.0],
    population_sizes=[10, 25, 50, 100, 200],
    n_trials=15,
    n_noise=200,
    signal_generator=generate_periodic,
    signal_kwargs_base={"period": 30.0, "n_epochs": 200},
    noise_kwargs={"n_epochs": 200},
    null_dist=null_dist,
    embedder=embedder,
    rng=np.random.default_rng(rng.integers(0, 2**63)),
    verbose=True,
)

In [ ]:
# Heatmap of detection rates
fig, ax = plt.subplots(figsize=(8, 6))
rates = pa_results["detection_rates"]
im = ax.imshow(rates, aspect="auto", cmap="RdYlGn", vmin=0, vmax=1, origin="lower")

ax.set_xticks(range(len(pa_results["population_sizes"])))
ax.set_xticklabels(pa_results["population_sizes"])
ax.set_yticks(range(len(pa_results["snr_levels"])))
ax.set_yticklabels([f"{s:.1f}" for s in pa_results["snr_levels"]])
ax.set_xlabel("Number of Injected Signal Sources")
ax.set_ylabel("Peak SNR (σ)")
ax.set_title("Detection Rate — Periodic Signals")

# Annotate cells
for i in range(rates.shape[0]):
    for j in range(rates.shape[1]):
        color = "white" if rates[i, j] < 0.5 else "black"
        ax.text(j, i, f"{rates[i, j]:.0%}", ha="center", va="center",
                color=color, fontsize=10, fontweight="bold")

plt.colorbar(im, ax=ax, label="Detection Rate", shrink=0.8)
fig.tight_layout()
plt.show()

In [ ]:
# Sensitivity curves: detection rate vs SNR for each population size
fig, ax = plt.subplots(figsize=(8, 5))
for j, n_sig in enumerate(pa_results["population_sizes"]):
    ax.plot(pa_results["snr_levels"], rates[:, j], "o-", label=f"N={n_sig}")

ax.axhline(0.8, color="gray", linestyle="--", alpha=0.5, label="80% threshold")
ax.axhline(0.5, color="gray", linestyle=":", alpha=0.5, label="50% threshold")
ax.set_xlabel("Peak SNR (σ)")
ax.set_ylabel("Detection Rate")
ax.set_title("Sensitivity Curves — Periodic Signals")
ax.legend(fontsize=8, ncol=2)
ax.set_ylim(-0.05, 1.05)
ax.grid(alpha=0.3)
fig.tight_layout()
plt.show()

## 4. Power analysis: transient signals

Repeat for supernova-like transients — a fundamentally different signal morphology.

In [ ]:
pa_transient = power_analysis(
    snr_levels=[1.0, 2.0, 3.0, 4.0, 5.0],
    population_sizes=[10, 50, 100, 200],
    n_trials=15,
    n_noise=200,
    signal_generator=generate_transient,
    signal_kwargs_base={"n_epochs": 200},
    noise_kwargs={"n_epochs": 200},
    null_dist=null_dist,
    embedder=embedder,
    rng=np.random.default_rng(rng.integers(0, 2**63)),
    verbose=True,
)

# Heatmap
fig, ax = plt.subplots(figsize=(7, 5))
rates_t = pa_transient["detection_rates"]
im = ax.imshow(rates_t, aspect="auto", cmap="RdYlGn", vmin=0, vmax=1, origin="lower")
ax.set_xticks(range(len(pa_transient["population_sizes"])))
ax.set_xticklabels(pa_transient["population_sizes"])
ax.set_yticks(range(len(pa_transient["snr_levels"])))
ax.set_yticklabels([f"{s:.1f}" for s in pa_transient["snr_levels"]])
ax.set_xlabel("Number of Injected Transients")
ax.set_ylabel("Peak SNR (σ)")
ax.set_title("Detection Rate — Transient Signals")
for i in range(rates_t.shape[0]):
    for j in range(rates_t.shape[1]):
        color = "white" if rates_t[i, j] < 0.5 else "black"
        ax.text(j, i, f"{rates_t[i, j]:.0%}", ha="center", va="center",
                color=color, fontsize=10, fontweight="bold")
plt.colorbar(im, ax=ax, label="Detection Rate", shrink=0.8)
fig.tight_layout()
plt.show()

## Summary

The power analysis answers the central feasibility question:

- **At what (SNR, N) does topological detection become reliable?**
- **Which topological statistics are most sensitive to which source types?**
- **What is the minimum population size for detection at sub-threshold SNR?**

These sensitivity curves define the operating envelope of the method and
inform the design of the real-data analysis in Sprints 3-4.